# Marketplace Pricing Intelligence

Text + structured pricing system with four modeling tracks and seller-facing inference path.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_prep import load_mercari_data, clean_mercari, split_data, build_pycaret_table
from src.features import build_preprocessor, prepare_feature_matrices
from src.modeling import (
    run_lazypredict_discovery,
    select_top3_eligible_families,
    run_manual_engineering_track,
    run_flaml_track,
    run_pycaret_track,
    category_error_analysis,
    seller_pricing_logic,
    save_inference_bundle,
    make_estimator,
    pricing_metrics,
)
from src.evaluation import build_leaderboard

SEED = 42
PROJECT_NAME = 'marketplace-pricing-intelligence'
PROJECT_ROOT = Path('.')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'mercari'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## 1) Business Problem and Success Criteria

Predict fair listing prices for sellers using listing text and metadata.

Primary metric: RMSLE
Secondary: MAE
Tertiary: latency/model practicality

## 2) Dataset Access and Data Dictionary

Dataset: Mercari Price Suggestion Challenge with title, description, category, brand, shipping, and condition.

In [ ]:
raw_df = load_mercari_data(RAW_DIR, sample_frac=0.06, random_state=SEED)
raw_df.shape

## 3) Data Cleaning and Leakage Checks

- Fill missing text/category/brand
- Remove invalid/zero prices
- Preserve realistic seller input columns

In [ ]:
df = clean_mercari(raw_df)
train_df, holdout_df = split_data(df, test_size=0.2, random_state=SEED)
train_df['price'].describe()

## 4) Feature Engineering

- Tfidf on title and description
- One-hot on category and brand
- Numeric shipping/condition features
- Reduced representation for LazyPredict

In [ ]:
preprocessor = build_preprocessor()
bundle = prepare_feature_matrices(train_df, holdout_df, preprocessor, random_state=SEED)
bundle['X_train_full'].shape, bundle['X_train_lazy'].shape

## 5) Validation Strategy

- Price-quantile stratified holdout split
- Stable metrics on original price scale
- Multi-seed rerun for top manual finalists

In [ ]:
pd.DataFrame({
    'split': ['train', 'holdout'],
    'rows': [len(train_df), len(holdout_df)],
    'mean_price': [train_df['price'].mean(), holdout_df['price'].mean()]
})

## 6) Track 1: LazyPredict Discovery Lab

Run on reduced feature representation for rapid family discovery.

In [ ]:
lazy_table = run_lazypredict_discovery(
    bundle['X_train_lazy'],
    bundle['X_holdout_lazy'],
    bundle['y_train_log'],
    bundle['y_holdout_price'],
)
lazy_table.head(15)

## 7) Selection of Top 3 Eligible Models

Only eligible top 3 families from LazyPredict proceed to manual lab.

In [ ]:
eligible_table, top3_families = select_top3_eligible_families(
    lazy_table=lazy_table,
    X_train_lazy=bundle['X_train_lazy'],
    y_train_log=bundle['y_train_log'],
    X_holdout_lazy=bundle['X_holdout_lazy'],
    y_holdout_price=bundle['y_holdout_price'],
    random_state=SEED,
)
eligible_table, top3_families

## 8) Track 2: Manual Engineering Lab

Manual top-3 modeling with category-level error analysis and seller-facing logic.

In [ ]:
manual_results, manual_models, manual_preds = run_manual_engineering_track(
    top3_families=top3_families,
    X_train_full=bundle['X_train_full'],
    y_train_log=bundle['y_train_log'],
    X_holdout_full=bundle['X_holdout_full'],
    y_holdout_price=bundle['y_holdout_price'],
    holdout_meta=bundle['holdout_meta'],
    random_state=SEED,
)
manual_results[['model_name', 'rmsle', 'mae', 'p95_latency_ms']]

In [ ]:
best_manual = manual_results.sort_values('rmsle').iloc[0]['model_name']
category_errors = category_error_analysis(bundle['holdout_meta'], manual_preds[best_manual])
category_errors.to_csv(ARTIFACT_DIR / 'category_error_analysis.csv', index=False)
category_errors.head(15)

## 9) Track 3: FLAML Optimization Lab

In [ ]:
flaml_result = run_flaml_track(
    X_train_full=bundle['X_train_full'],
    y_train_log=bundle['y_train_log'],
    X_holdout_full=bundle['X_holdout_full'],
    y_holdout_price=bundle['y_holdout_price'],
    time_budget=150,
    random_state=SEED,
)
flaml_result

## 10) Track 4: PyCaret Experiment Lab

PyCaret workflow on engineered tabular surrogate features for deployability comparison.

In [ ]:
pycaret_train = build_pycaret_table(train_df)
pycaret_holdout = build_pycaret_table(holdout_df)
pycaret_result = run_pycaret_track(
    train_table=pycaret_train,
    holdout_table=pycaret_holdout,
    session_id=SEED,
    model_output_path=ARTIFACT_DIR / 'pycaret_marketplace_pricing',
)
pycaret_result

## 11) Unified Leaderboard and Final Model Ranking

Compares LazyPredict top candidates, manual top-3, FLAML winner, and PyCaret winner.

In [ ]:
leaderboard = build_leaderboard(
    project_name=PROJECT_NAME,
    lazy_results=eligible_table,
    manual_results=manual_results,
    flaml_result=flaml_result,
    pycaret_result=pycaret_result,
)
leaderboard.head(10)

In [ ]:
leaderboard.to_csv(ARTIFACT_DIR / 'leaderboard_marketplace_pricing.csv', index=False)
leaderboard[['model_name', 'library_source', 'holdout_primary_metric', 'holdout_secondary_metric', 'final_rank']].head(10)

In [ ]:
# Re-run top manual candidates across multiple seeds for stability check
seed_rows = []
for model_name in manual_results.sort_values('rmsle').head(3)['model_name'].tolist():
    for seed in [42, 123, 777]:
        model = make_estimator(model_name, random_state=seed)
        model.fit(bundle['X_train_full'], bundle['y_train_log'])
        pred_price = np.expm1(model.predict(bundle['X_holdout_full'])).clip(min=0)
        m = pricing_metrics(bundle['y_holdout_price'], pred_price)
        seed_rows.append({'model_name': model_name, 'seed': seed, 'rmsle': m['rmsle'], 'mae': m['mae']})
seed_df = pd.DataFrame(seed_rows)
seed_df.to_csv(ARTIFACT_DIR / 'rerun_seed_stability.csv', index=False)
seed_df

## 12) Business Recommendation

Document winning model preference, safer second-best context, and rejected tradeoff after execution.

## 13) Inference / Deployment Path

Persist deployable artifact and seller-facing price logic.

In [ ]:
winner = leaderboard.sort_values('final_rank').iloc[0]
if winner['library_source'] == 'manual' and winner['model_name'] in manual_models:
    winner_model = manual_models[winner['model_name']]
    pred = manual_preds[winner['model_name']]
else:
    # fallback to best manual model for API artifact
    fallback_name = manual_results.sort_values('rmsle').iloc[0]['model_name']
    winner_model = manual_models[fallback_name]
    pred = manual_preds[fallback_name]

residuals = bundle['y_holdout_price'] - pred
save_inference_bundle(winner_model, preprocessor, residuals, ARTIFACT_DIR)
seller_pricing_logic(float(np.median(pred)), float(np.mean(np.abs(residuals))))

## 14) Monitoring / Drift / Retraining Plan

Monitor category-share drift, price distribution shift, RMSLE drift, and residual quantile drift.

## 15) Limitations and Next Steps

- Add multi-output interval prediction
- Add separate models for major category clusters
- Add feedback loop from sold-price outcomes